<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/fluidflow/finite_element_methods_oil_gas_neqsim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finite-element methods for oil & gas engineering with NeqSim

This notebook establishes a reusable open-source stack: **NeqSim → Gmsh → scikit-fem / FEniCSx → PyVista**, while retaining OpenFOAM for CFD.

- **NeqSim** owns thermodynamics, phase equilibrium and fluid transport properties.
- **scikit-fem** is used for compact, transparent FEM models.
- **Gmsh** creates non-uniform geometry, meshes and physical groups.
- **FEniCSx** solves the more general PDE field problems.
- **PyVista** provides a common VTK-compatible mesh/field representation.

NeqSim is executed in an isolated subprocess and its results are serialized before native FEM libraries are loaded. The executable examples cover insulated-pipe heat transfer, damaged insulation with Gmsh, NeqSim-derived porous-gas diffusion, and wellbore-to-formation heat conduction. The companion `neqsim_fenicsx_fem_pipeline.ipynb` adds transient cooldown, hydrate margin and thermo-elastic pipe stress.

In [ ]:
import hashlib, importlib.util, os, shutil, subprocess, sys
from pathlib import Path

NEQSIM_SOURCE_REF = 'master'
os.environ['NEQSIM_JVM_AUTOSTART'] = '0'

def rq(cmd, cwd=None):
    return subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True).stdout

rq([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim', 'scikit-fem', 'gmsh', 'meshio', 'pyvista', 'scipy'])
if importlib.util.find_spec('dolfinx') is None:
    installer = Path('/tmp/fenicsx.sh')
    rq(['wget', '-q', 'https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh', '-O', str(installer)])
    rq(['bash', str(installer)])

src = Path('/content/neqsim-java')
if src.exists():
    shutil.rmtree(src)
rq(['git', 'clone', '--depth', '1', '--branch', NEQSIM_SOURCE_REF, 'https://github.com/equinor/neqsim.git', str(src)])
neqsim_commit = rq(['git', '-C', str(src), 'rev-parse', 'HEAD']).strip()
rq(['./mvnw', '-q', '-DskipTests', '-P', 'shade', 'package'], cwd=src)
candidates = [p for p in (src/'target').glob('neqsim-*.jar') if '-sources' not in p.name and '-javadoc' not in p.name and not p.name.startswith('original-')]
if not candidates:
    raise FileNotFoundError('No NeqSim runtime JAR found')
neqsim_jar = max(candidates, key=lambda p: p.stat().st_size)
assert neqsim_jar.stat().st_size > 5_000_000
neqsim_jar_sha256 = hashlib.sha256(neqsim_jar.read_bytes()).hexdigest()

print('NeqSim master commit:', neqsim_commit)
print('NeqSim runtime JAR:', neqsim_jar)
print('JAR SHA-256:', neqsim_jar_sha256)
print('FEniCSx installed:', importlib.util.find_spec('dolfinx') is not None)

## Common NeqSim state and transport-property handoff

One natural-gas state is evaluated in an isolated NeqSim JVM. The handoff contains density, viscosity, thermal conductivity, mass-specific heat capacity and an effective multicomponent CO₂ diffusion coefficient. The Java process exits before scikit-fem, Gmsh, FEniCSx or PyVista are imported.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Tg, Pg = 45.0, 75.0

neqsim_worker = r'''
import json, os
from pathlib import Path
import jpype
jar = os.environ['NEQSIM_JAR']
out = Path(os.environ['NEQSIM_RESULTS'])
jpype.startJVM('-Xrs', classpath=[jar], convertStrings=False, interrupt=False)
SystemSrkEos = jpype.JClass('neqsim.thermo.system.SystemSrkEos')
Ops = jpype.JClass('neqsim.thermodynamicoperations.ThermodynamicOperations')
MainOnly = jpype.JClass('neqsim.pvtsimulation.flowassurance.SurfCooldownAnalyzer')
source_location = str(MainOnly.class_.getProtectionDomain().getCodeSource().getLocation())
assert Path(jar).name in source_location
Tg, Pg = 45.0, 75.0
composition = {'nitrogen':0.01, 'CO2':0.03, 'methane':0.84, 'ethane':0.07, 'propane':0.03, 'i-butane':0.005, 'n-butane':0.01, 'n-pentane':0.005}
fluid = SystemSrkEos(Tg+273.15, Pg)
for name, value in composition.items():
    fluid.addComponent(name, value)
fluid.setMixingRule('classic')
Ops(fluid).TPflash()
fluid.initPhysicalProperties()
gas = fluid.getPhase('gas')
physical = gas.getPhysicalProperties()
physical.setDiffusionCoefficientModel('Fuller-Schettler-Giddings')
gas.initPhysicalProperties()
physical.calcEffectiveDiffusionCoefficients()
result = {
    'source_location': source_location,
    'rho': float(gas.getDensity('kg/m3')),
    'mu': float(gas.getViscosity('kg/msec')),
    'k': float(gas.getThermalConductivity('W/mK')),
    'cp': float(gas.getCp('J/kgK')),
    'D_CO2': float(physical.getEffectiveDiffusionCoefficient('CO2')),
}
out.write_text(json.dumps(result, indent=2))
'''

handoff_path = Path('/tmp/neqsim_fem_stack_handoff.json')
env = os.environ.copy()
env['NEQSIM_JAR'] = str(neqsim_jar)
env['NEQSIM_RESULTS'] = str(handoff_path)
subprocess.run([sys.executable, '-c', neqsim_worker], env=env, check=True)
properties = json.loads(handoff_path.read_text())
assert Path(neqsim_jar).name in properties.pop('source_location')
assert all(np.isfinite(v) and v > 0 for v in properties.values())
display(pd.DataFrame({'property': list(properties), 'value': list(properties.values())}))

## A. scikit-fem — lightweight radial insulated-pipe heat transfer

Start with scikit-fem when geometry is simple and transparency matters. The mesh is aligned with the steel/insulation interface and the axisymmetric weak-form result is checked directly against the analytical cylindrical thermal-resistance solution.

In [ ]:
from skfem import Basis, BilinearForm, FacetBasis, LinearForm, MeshLine, MeshTri, ElementLineP1, ElementTriP1, asm, condense, solve
from skfem.helpers import dot, grad

Di, ts, ti = 0.254, 0.0127, 0.050
ri, rs, ro = Di/2, Di/2+ts, Di/2+ts+ti
ks, ki, Tsea, ho, velocity = 50.0, 0.17, 4.0, 300.0, 5.0
Re = properties['rho']*velocity*Di/properties['mu']
Pr = properties['cp']*properties['mu']/properties['k']
friction = (0.79*np.log(Re)-1.64)**-2
Nu = (friction/8)*(Re-1000)*Pr/(1+12.7*np.sqrt(friction/8)*(Pr**(2/3)-1))
hi = Nu*properties['k']/Di

nodes = np.r_[np.linspace(ri, rs, 31), np.linspace(rs, ro, 91)[1:]]
line = MeshLine(nodes)
element = ElementLineP1()
basis = Basis(line, element)
inner_facets = FacetBasis(line, element, facets=line.facets_satisfying(lambda xx: np.isclose(xx[0], ri)))
outer_facets = FacetBasis(line, element, facets=line.facets_satisfying(lambda xx: np.isclose(xx[0], ro)))
@BilinearForm
def conduction(u, v, w): return np.where(w.x[0] <= rs, ks, ki)*w.x[0]*dot(grad(u), grad(v))
@BilinearForm
def robin_inner(u, v, w): return hi*w.x[0]*u*v
@BilinearForm
def robin_outer(u, v, w): return ho*w.x[0]*u*v
@LinearForm
def load_inner(v, w): return hi*w.x[0]*(Tg+273.15)*v
@LinearForm
def load_outer(v, w): return ho*w.x[0]*(Tsea+273.15)*v
A = asm(conduction, basis) + asm(robin_inner, inner_facets) + asm(robin_outer, outer_facets)
rhs = asm(load_inner, inner_facets) + asm(load_outer, outer_facets)
Tk = solve(A, rhs) - 273.15
rr = line.p[0]
R = 1/(hi*2*np.pi*ri) + np.log(rs/ri)/(2*np.pi*ks) + np.log(ro/rs)/(2*np.pi*ki) + 1/(ho*2*np.pi*ro)
q_analytic = (Tg-Tsea)/R
T_analytic = Tg - q_analytic/(hi*2*np.pi*ri)
T_skfem = float(Tk[np.argmin(abs(rr-ri))])
print(f'scikit-fem inner wall: {T_skfem:.4f} degC')
print(f'analytical inner wall: {T_analytic:.4f} degC')
assert abs(T_skfem-T_analytic) < 0.2
plt.plot(rr, Tk); plt.axvline(rs, ls='--', label='steel/insulation'); plt.xlabel('Radius [m]'); plt.ylabel('Temperature [degC]'); plt.legend(); plt.grid(); plt.show()

## B. Gmsh → FEniCSx → PyVista — damaged insulation

Gmsh becomes valuable when geometry is non-uniform. Here a 0.4 m local section loses half of the insulation thickness. Gmsh creates physical groups for steel, insulation, the inner wall and the exposed seawater boundary. DOLFINx imports those tags directly.

PyVista is used as a VTK-compatible mesh/field container while Matplotlib renders the field, avoiding a dependency on an OpenGL display.

In [ ]:
import gmsh
import pyvista as pv
import ufl
from mpi4py import MPI
from dolfinx import fem, plot
from dolfinx.io import gmsh as gmshio
from dolfinx.fem.petsc import LinearProblem

assert MPI.COMM_WORLD.size == 1
L, a0, a1, loss = 2.0, 0.8, 1.2, 0.5*ti
gmsh.initialize()
try:
    gmsh.option.setNumber('General.Terminal', 0)
    gmsh.model.add('damaged_pipe')
    occ = gmsh.model.occ
    steel_surface = occ.addRectangle(0, ri, 0, L, ts)
    insulation_surface = occ.addRectangle(0, rs, 0, L, ti)
    notch = occ.addRectangle(a0, ro-loss, 0, a1-a0, loss)
    insulation_cut, _ = occ.cut([(2, insulation_surface)], [(2, notch)], removeObject=True, removeTool=True)
    occ.fragment([(2, steel_surface)], insulation_cut)
    occ.synchronize()
    surfaces = [tag for dim, tag in gmsh.model.getEntities(2)]
    steel_surfaces, insulation_surfaces = [], []
    for surface in surfaces:
        _, rc, _ = occ.getCenterOfMass(2, surface)
        (steel_surfaces if rc < rs else insulation_surfaces).append(surface)
    gmsh.model.addPhysicalGroup(2, steel_surfaces, 1)
    gmsh.model.addPhysicalGroup(2, insulation_surfaces, 2)
    boundary = gmsh.model.getBoundary([(2,s) for s in surfaces], combined=True, oriented=False)
    inner_curves, outer_curves, end_curves = [], [], []
    for dim, curve in boundary:
        zc, rc, _ = occ.getCenterOfMass(dim, curve)
        if np.isclose(rc, ri, atol=1e-7): inner_curves.append(curve)
        elif np.isclose(zc, 0.0, atol=1e-7) or np.isclose(zc, L, atol=1e-7): end_curves.append(curve)
        else: outer_curves.append(curve)
    gmsh.model.addPhysicalGroup(1, inner_curves, 11)
    gmsh.model.addPhysicalGroup(1, outer_curves, 12)
    if end_curves: gmsh.model.addPhysicalGroup(1, end_curves, 13)
    gmsh.option.setNumber('Mesh.CharacteristicLengthMin', 0.012)
    gmsh.option.setNumber('Mesh.CharacteristicLengthMax', 0.04)
    gmsh.model.mesh.generate(2)
    imported = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, 0, gdim=2)
finally:
    gmsh.finalize()

domain = imported.mesh if hasattr(imported, 'mesh') else imported[0]
cell_tags = imported.cell_tags if hasattr(imported, 'cell_tags') else imported[1]
facet_tags = imported.facet_tags if hasattr(imported, 'facet_tags') else imported[2]
V = fem.functionspace(domain, ('Lagrange',1))
u, vtest = ufl.TrialFunction(V), ufl.TestFunction(V)
X = ufl.SpatialCoordinate(domain); r = X[1]
dx = ufl.Measure('dx', domain=domain, subdomain_data=cell_tags)
ds = ufl.Measure('ds', domain=domain, subdomain_data=facet_tags)
a = ks*ufl.inner(ufl.grad(u),ufl.grad(vtest))*2*np.pi*r*dx(1) + ki*ufl.inner(ufl.grad(u),ufl.grad(vtest))*2*np.pi*r*dx(2) + hi*u*vtest*2*np.pi*r*ds(11) + ho*u*vtest*2*np.pi*r*ds(12)
Lform = hi*(Tg+273.15)*vtest*2*np.pi*r*ds(11) + ho*(Tsea+273.15)*vtest*2*np.pi*r*ds(12)
gmsh_problem = LinearProblem(a, Lform, petsc_options_prefix='gmsh_heat_', petsc_options={'ksp_type':'cg','pc_type':'jacobi','ksp_rtol':1e-10})
uh = gmsh_problem.solve()
assert gmsh_problem.solver.getConvergedReason() > 0
coords = V.tabulate_dof_coordinates()
inner_wall = (uh.x.array-273.15)[np.isclose(coords[:,1],ri)]
print('Local minimum inner-wall T:', float(inner_wall.min()), 'degC')
assert uh.x.array.min() >= Tsea+273.15-1e-6 and uh.x.array.max() <= Tg+273.15+1e-6

cells, cell_types, points = plot.vtk_mesh(V)
grid = pv.UnstructuredGrid(cells, cell_types, points)
grid.point_data['Temperature [degC]'] = uh.x.array-273.15
assert grid.n_points == len(uh.x.array)
print(grid)
plt.figure(figsize=(10,3))
sc = plt.scatter(points[:,0], points[:,1], c=uh.x.array-273.15, s=10)
plt.colorbar(sc, label='Temperature [degC]')
plt.xlabel('Axial coordinate [m]'); plt.ylabel('Radius [m]'); plt.title('Gmsh → FEniCSx field stored in PyVista'); plt.show()

## C. NeqSim diffusion → porous-rock FEM

The NeqSim handoff provides a multicomponent molecular CO₂ diffusion coefficient. A porous-medium screen applies $D_{rock}=\phi D_{mol}/\tau$. Porosity and tortuosity are rock-model inputs, not thermodynamic properties. scikit-fem then solves transient 2D diffusion.

In [ ]:
phi, tortuosity = 0.22, 2.5
Dmol = properties['D_CO2']
Drock = phi/tortuosity*Dmol
print(f'NeqSim molecular D_CO2: {Dmol:.3e} m2/s')
print(f'Porous effective D_CO2: {Drock:.3e} m2/s')
rock_mesh = MeshTri.init_tensor(np.linspace(0,2,61), np.linspace(0,1,31))
rock_basis = Basis(rock_mesh, ElementTriP1())
@BilinearForm
def mass(u,v,w): return u*v
@BilinearForm
def diffusion(u,v,w): return Drock*dot(grad(u),grad(v))
M = asm(mass, rock_basis); K = asm(diffusion, rock_basis)
dt = 6*3600.0
left = rock_basis.get_dofs(lambda xx: np.isclose(xx[0],0)).all()
concentration = np.zeros(rock_basis.N); snapshots = {}
for n in range(80):
    prescribed = np.zeros(rock_basis.N); prescribed[left] = 1.0
    Ac, bc, x0, free = condense(M+dt*K, M@concentration, x=prescribed, D=left)
    concentration = x0.copy(); concentration[free] = solve(Ac, bc)
    if n in (9,39,79): snapshots[(n+1)*dt/86400] = concentration.copy()
xy = rock_mesh.p
mid = np.where(np.isclose(xy[1],0.5,atol=0.02))[0]
order = np.argsort(xy[0,mid])
for days, values in snapshots.items(): plt.plot(xy[0,mid][order], values[mid][order], label=f'{days:.1f} d')
plt.xlabel('Distance [m]'); plt.ylabel('Normalized CO2'); plt.legend(); plt.grid(); plt.show()
assert concentration.min() >= -1e-6 and concentration.max() <= 1+1e-6

## D. FEniCSx wellbore-to-formation heat conduction

A 200 m well segment is embedded in a radial rock domain with a geothermal far-field trend. The well boundary uses a convection coefficient derived from the same NeqSim fluid. This demonstrates why FEniCSx is useful when the engineering output is a 2D temperature field rather than only a scalar outlet condition.

In [ ]:
from dolfinx import mesh

H, rw, rf = 200.0, 0.10, 15.0
kr, hfar = 2.5, 50.0
hw = min(hi, 1500.0)
Tfluid, Tfar0, geothermal_gradient = 80.0, 25.0, 0.03
well_domain = mesh.create_rectangle(MPI.COMM_WORLD, np.array([[0.0,rw],[H,rf]]), [70,40], cell_type=mesh.CellType.triangle)
fdim = well_domain.topology.dim-1
well_facets = mesh.locate_entities_boundary(well_domain, fdim, lambda xx: np.isclose(xx[1],rw))
far_facets = mesh.locate_entities_boundary(well_domain, fdim, lambda xx: np.isclose(xx[1],rf))
entities = np.hstack([well_facets,far_facets]).astype(np.int32)
values = np.hstack([np.ones(len(well_facets),np.int32),2*np.ones(len(far_facets),np.int32)])
order = np.argsort(entities)
tags = mesh.meshtags(well_domain, fdim, entities[order], values[order])
Vw = fem.functionspace(well_domain, ('Lagrange',1))
u, vtest = ufl.TrialFunction(Vw), ufl.TestFunction(Vw)
X = ufl.SpatialCoordinate(well_domain); zcoord, rcoord = X[0], X[1]
ds = ufl.Measure('ds', domain=well_domain, subdomain_data=tags); dx = ufl.Measure('dx', domain=well_domain)
Tfar = Tfar0+273.15+geothermal_gradient*zcoord
a = kr*ufl.inner(ufl.grad(u),ufl.grad(vtest))*2*np.pi*rcoord*dx + hw*u*vtest*2*np.pi*rcoord*ds(1) + hfar*u*vtest*2*np.pi*rcoord*ds(2)
Lform = hw*(Tfluid+273.15)*vtest*2*np.pi*rcoord*ds(1) + hfar*Tfar*vtest*2*np.pi*rcoord*ds(2)
well_problem = LinearProblem(a, Lform, petsc_options_prefix='well_heat_', petsc_options={'ksp_type':'cg','pc_type':'jacobi','ksp_rtol':1e-10})
Tw = well_problem.solve()
assert well_problem.solver.getConvergedReason() > 0
coords = Vw.tabulate_dof_coordinates()
wall_temperature = (Tw.x.array-273.15)[np.isclose(coords[:,1],rw)]
print(f'Well-wall temperature range: {float(wall_temperature.min()):.2f} to {float(wall_temperature.max()):.2f} degC')
cells, cell_types, points = plot.vtk_mesh(Vw)
well_grid = pv.UnstructuredGrid(cells, cell_types, points)
well_grid.point_data['Temperature [degC]'] = Tw.x.array-273.15
assert well_grid.n_points == len(Tw.x.array)
print(well_grid)
plt.figure(figsize=(8,5))
sc = plt.scatter(points[:,0], points[:,1], c=Tw.x.array-273.15, s=8)
plt.colorbar(sc, label='Temperature [degC]')
plt.xlabel('Depth coordinate [m]'); plt.ylabel('Radius [m]'); plt.title('Well/formation field stored in PyVista'); plt.show()

## Recommended hierarchy

- **NeqSim**: thermodynamics, process and flow boundary conditions.
- **scikit-fem**: simple, transparent FEM and rapid Colab demonstrations.
- **Gmsh**: geometry, physical groups and reusable meshing.
- **FEniCSx**: detailed heat, mechanics, diffusion and multiphysics PDEs.
- **PyVista**: common VTK-compatible mesh/field representation.
- **OpenFOAM**: CFD when momentum, velocity and pressure fields dominate.

The companion pipeline notebook extends this stack to shutdown cooldown, hydrate no-touch time and thermo-elastic stress. Natural later extensions are separator/nozzle thermal stress, buried-pipeline soil domains, electrochemistry/corrosion and coupled geomechanics.